# 00 — Current WDM–LISA response pipeline

This is the maintained explanatory notebook. It uses the physical
`esa-trailing-orbits.h5` orbit file, which must be placed in the same directory
as the notebooks.

The notebook follows the current staged implementation:

\[
\mathrm{A0:\ geometry\ and\ prepared\ operators}
\rightarrow
\mathrm{A1:\ waveform\ DWTs\ and\ six\ one\!-\!arm\ links}
\rightarrow
\mathrm{B:\ prepared\ Gen\!-\!2\ Michelson}\ (X,Y,Z).
\]

The example is intentionally short, but it exercises the same physical orbit
loader and response code used by the larger profiling notebooks.

Optional output is saved directly beside the notebook with a `_local_wdm_`
prefix.

In [1]:

import json
import platform
import subprocess
import time
from pathlib import Path

import jax
import numpy as np

from lisaorbits import InterpolatedOrbits
from lisaorbits.file import read_orbit_file

import WDM_LISAresponse


NOTEBOOK_DIR = Path.cwd().resolve()
ORBIT_FILE = NOTEBOOK_DIR / "esa-trailing-orbits.h5"

if not ORBIT_FILE.exists():
    raise FileNotFoundError(
        "Place esa-trailing-orbits.h5 in the same directory as these notebooks. "
        f"Current notebook directory: {NOTEBOOK_DIR}"
    )


def load_lisa_orbits(path: Path):
    """Load the physical LISA orbit file used by the response calculations."""
    orbit_data = read_orbit_file(str(path))
    orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
    orbits = InterpolatedOrbits(
        orbit_times,
        orbit_data.positions,
        orbit_data.velocities,
    )
    return orbit_data, orbits


def git_state(package_file: str) -> dict[str, object]:
    """Return the current Git commit and whether the checkout is dirty."""
    completed = subprocess.run(
        [
            "git",
            "-C",
            str(Path(package_file).resolve().parent),
            "rev-parse",
            "--show-toplevel",
        ],
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        return {"root": None, "commit": None, "dirty": None}

    root = Path(completed.stdout.strip()).resolve()
    commit = subprocess.run(
        ["git", "-C", str(root), "rev-parse", "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    dirty = bool(
        subprocess.run(
            ["git", "-C", str(root), "status", "--porcelain"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    )
    return {"root": str(root), "commit": commit, "dirty": dirty}


def json_ready(value):
    """Convert NumPy and JAX objects into JSON-compatible values."""
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if hasattr(value, "item") and getattr(value, "ndim", None) == 0:
        return value.item()
    if hasattr(value, "tolist"):
        return value.tolist()
    return value


def save_json(filename: str, payload) -> Path:
    """Save a JSON result beside the notebook."""
    path = NOTEBOOK_DIR / filename
    path.write_text(
        json.dumps(json_ready(payload), indent=2),
        encoding="utf-8",
    )
    print("Saved:", path)
    return path


orbit_data, orbits = load_lisa_orbits(ORBIT_FILE)
YEAR_SI = 365.25 * 24.0 * 3600.0
t0 = float(orbit_data.t0 + YEAR_SI)

print("Notebook directory:", NOTEBOOK_DIR)
print("Orbit file:", ORBIT_FILE)
print("Python:", platform.python_version())
print("JAX:", jax.__version__, "| backend:", jax.default_backend())
print("WDM_LISAresponse:", WDM_LISAresponse.__file__)
print("Source state:", git_state(WDM_LISAresponse.__file__))


from WDM_LISAresponse.code.lisa_response.TDI_with_one_arm_plan import (
    gen2_michelson_wdm_from_waveform_precomputed,
    precompute_gen2_michelson_wdm_geometry,
    precompute_gen2_michelson_wdm_waveform,
)
from WDM_LISAresponse.code.lisa_response.waveforms import (
    generate_windowed_padded_waveform,
)


Notebook directory: /Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse/examples
Orbit file: /Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse/examples/esa-trailing-orbits.h5
Python: 3.13.11
JAX: 0.7.0 | backend: cpu
WDM_LISAresponse: /Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse/src/WDM_LISAresponse/__init__.py
Source state: {'root': '/Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse', 'commit': '155979a0a509bc82690c867631655d2434783661', 'dirty': True}


## 1. Compact demonstration configuration

The WDM transform uses a regular \((n,m)\) grid. The static A0 stage depends
on the grid, orbit model, sky position and shift settings, but not on the
waveform coefficients.

`calc_m0=False` is the production convention. Numerical comparisons later
in the project should therefore use the physical columns `[..., 1:]`.


In [ ]:
N = 65_536
dt = 10.0
Nf = 64
q_wdm = 4
d_wdm = 4
A_frac = 0.25
L_TRUNC = 25
CALC_M0 = False

theta = 0.9
phi = 0.4

SHIFT_KWARGS = {
    "assembly_backend": "production",
    "assembly_precision": "complex64",
    "row_chunk_size": 512,
    "lag_block_size": 8,
    "batch_chunk": 32,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 16,
    "tl_tp_interp_kind": "linear",
}

SAVE_SUMMARY = False

times = t0 + np.arange(N, dtype=float) * dt

print("Time samples:", times.size)
print("Start time:", times[0])
print("Nominal duration [days]:", times.size * dt / 86_400.0)
print("Shift settings:", SHIFT_KWARGS)

Time samples: 65536
Start time: 2104768730.8175
Nominal duration [days]: 7.5851851851851855
Shift settings: {'assembly_backend': 'production', 'assembly_precision': 'complex64', 'row_chunk_size': 512, 'lag_block_size': 8, 'batch_chunk': 32, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 16, 'tl_tp_interp_kind': 'linear'}


## 2. Generate the time-domain waveform

The waveform generator and windowing/padding utilities are independent of
the response operator. Here we use a monochromatic signal so the notebook
remains quick and reproducible.


In [3]:
times_pad, h_plus, h_cross, padding = generate_windowed_padded_waveform(
    waveform_model="monochromatic",
    t0=t0,
    dt=dt,
    N=N,
    model_kwargs={
        "A_plus": 1.0e-21,
        "A_cross": 0.7e-21,
        "f": 2.0e-3,
        "phi_0": 0.3,
    },
    window_function="tukey",
    window_kwargs={"alpha": 0.01, "tukey_factor": 2.0},
    wdm_Nf=Nf,
    return_padding_info=True,
)

print("Original samples:", N)
print("Padded samples:", times_pad.size)
print("Expected WDM shape:", (padding["wdm_Nt"], Nf))
print("Waveform dtypes:", h_plus.dtype, h_cross.dtype)


Original samples: 65536
Padded samples: 66944
Expected WDM shape: (1046, 64)
Waveform dtypes: float64 float64


## 3. A0 — waveform-independent geometry and operators

A0 constructs:

- the padded WDM transform and its regular grid;
- six one-arm geometry/amplitude arrays;
- six prepared variable-delay WDM operators;
- recursively composed Gen-2 Michelson delay fields;
- one prepared 42-job TDI operator.

A0 is reusable for every waveform with the same grid, orbit model and sky
position.


In [4]:
a0_start = time.perf_counter()
geometry = precompute_gen2_michelson_wdm_geometry(
    times_pad,
    theta,
    phi,
    orbits,
    Nf=Nf,
    L_trunc=L_TRUNC,
    q=q_wdm,
    d=d_wdm,
    A_frac=A_frac,
    calc_m0=CALC_M0,
    variable_shift_kwargs=SHIFT_KWARGS,
)
a0_seconds = time.perf_counter() - a0_start

one_arm_plan = geometry["one_arm_plan"]
tdi_plan = geometry["tdi_plan"]

print("A0 seconds:", f"{a0_seconds:.3f}")
print("WDM shape:", (geometry["wdm"].Nt, geometry["wdm"].Nf))
print("One-arm jobs:", one_arm_plan.shift_plan.num_jobs)
print("TDI jobs:", tdi_plan.shift_plan.num_jobs)
print(
    "One-arm plan memory [MiB]:",
    one_arm_plan.shift_plan.device_plan_memory_bytes / 1024**2,
)
print(
    "TDI plan memory [MiB]:",
    tdi_plan.shift_plan.device_plan_memory_bytes / 1024**2,
)
print("Prepared backend:", one_arm_plan.shift_plan.resolved_assembly_backend)


A0 seconds: 2.512
WDM shape: (1046, 64)
One-arm jobs: 6
TDI jobs: 42
One-arm plan memory [MiB]: 1.6519966125488281
TDI plan memory [MiB]: 11.563587188720703
Prepared backend: production


## 4. A1 — transform one waveform and evaluate six one-arm links

A1 performs the waveform-dependent work:

1. pad the two polarizations onto the A0 WDM grid;
2. compute two WDM transforms;
3. form the six unshifted link coefficient arrays;
4. apply the six prepared variable-delay operators.

The maintained one-arm paths are `materialized` and
`materialized_parallel_links`. This walkthrough uses the simpler
`materialized` path; profiling notebooks test CPU scheduling separately.


In [5]:
a1_start = time.perf_counter()
waveform_state = precompute_gen2_michelson_wdm_waveform(
    geometry,
    h_plus,
    h_cross,
    return_device_links=True,
    one_arm_execution_mode="materialized",
    parallel_wdm_transforms=False,
)
a1_seconds = time.perf_counter() - a1_start

link_w = waveform_state["link_w"]
for value in link_w.values():
    value.block_until_ready()

print("A1 seconds:", f"{a1_seconds:.3f}")
print("Directed links:", tuple(link_w))
for link, values in link_w.items():
    print(
        f"  {link}: shape={tuple(values.shape)}, "
        f"dtype={values.dtype}, "
        f"physical_norm={float(jax.numpy.linalg.norm(values[..., 1:])):.3e}"
    )


A1 seconds: 0.618
Directed links: ((2, 1), (3, 2), (1, 3), (3, 1), (2, 3), (1, 2))
  (2, 1): shape=(1046, 64), dtype=float32, physical_norm=3.184e-10
  (3, 2): shape=(1046, 64), dtype=float32, physical_norm=3.253e-11
  (1, 3): shape=(1046, 64), dtype=float32, physical_norm=4.337e-10
  (3, 1): shape=(1046, 64), dtype=float32, physical_norm=4.333e-10
  (2, 3): shape=(1046, 64), dtype=float32, physical_norm=3.264e-11
  (1, 2): shape=(1046, 64), dtype=float32, physical_norm=3.180e-10


## 5. B — prepared second-generation Michelson TDI

The production B stage gathers the six directed links, applies the prepared
42-job operator, and accumulates directly into \(X,Y,Z\). It does not
reconstruct the delay kernels for every waveform.


In [6]:
b_start = time.perf_counter()

X, Y, Z = gen2_michelson_wdm_from_waveform_precomputed(
    waveform_state,
    tdi_assembly_mode="prepared_plan",
)

# The current prepared TDI path returns NumPy arrays, which are already
# synchronized. Keep this guard compatible with a future device-resident path.
for value in jax.tree_util.tree_leaves((X, Y, Z)):
    if hasattr(value, "block_until_ready"):
        value.block_until_ready()

b_seconds = time.perf_counter() - b_start

print("B seconds:", f"{b_seconds:.3f}")
for name, values in zip(("X", "Y", "Z"), (X, Y, Z)):
    print(
        f"{name}: shape={tuple(values.shape)}, dtype={values.dtype}, "
        f"physical_norm={float(jax.numpy.linalg.norm(values[..., 1:])):.3e}"
    )

B seconds: 1.224
X: shape=(1046, 64), dtype=float32, physical_norm=3.686e-11
Y: shape=(1046, 64), dtype=float32, physical_norm=5.625e-11
Z: shape=(1046, 64), dtype=float32, physical_norm=7.327e-11


## 6. What is reused in parameter estimation?

For a fixed detector geometry and sky position:

- **A0 is built once**;
- **A1 is repeated for every waveform proposal**;
- **B is repeated for every set of one-arm link coefficients**.

Sky-dependent inference will eventually require rebuilding or dynamically
updating part of A0. The current notebook documents the fixed-sky dense
implementation that serves as the validated baseline.


In [7]:
summary = {
    "source": git_state(Path(WDM_LISAresponse.__file__)),
    "configuration": {
        "N": N,
        "dt": dt,
        "Nf": Nf,
        "q_wdm": q_wdm,
        "L_trunc": L_TRUNC,
        "calc_m0": CALC_M0,
        "shift_kwargs": SHIFT_KWARGS,
    },
    "timings_s": {
        "A0": a0_seconds,
        "A1": a1_seconds,
        "B": b_seconds,
    },
    "shapes": {
        "wdm": [int(geometry["wdm"].Nt), int(geometry["wdm"].Nf)],
        "X": list(X.shape),
    },
}

print(json.dumps(json_ready(summary), indent=2))


if SAVE_SUMMARY:
    save_json(
        "_local_wdm_00_walkthrough_summary.json",
        summary,
    )


{
  "source": {
    "root": "/Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse",
    "commit": "155979a0a509bc82690c867631655d2434783661",
    "dirty": true
  },
  "configuration": {
    "N": 65536,
    "dt": 10.0,
    "Nf": 64,
    "q_wdm": 4,
    "L_trunc": 8,
    "calc_m0": false,
    "shift_kwargs": {
      "assembly_backend": "production",
      "assembly_precision": "complex64",
      "row_chunk_size": 512,
      "lag_block_size": 8,
      "batch_chunk": 32,
      "tl_tp_mode": "interp",
      "tl_tp_interp_points": 16,
      "tl_tp_interp_kind": "linear"
    }
  },
  "timings_s": {
    "A0": 2.5116530829982366,
    "A1": 0.6182304590183776,
    "B": 1.2237588339776266
  },
  "shapes": {
    "wdm": [
      1046,
      64
    ],
    "X": [
      1046,
      64
    ]
  }
}


In [8]:
expected_shape = (geometry["wdm"].Nt, geometry["wdm"].Nf)

assert expected_shape == (1046, 64)
assert one_arm_plan.shift_plan.num_jobs == 6
assert tdi_plan.shift_plan.num_jobs == 42

assert set(link_w) == {
    (2, 1),
    (3, 2),
    (1, 3),
    (3, 1),
    (2, 3),
    (1, 2),
}

for link, values in link_w.items():
    values_np = np.asarray(values)

    assert values_np.shape == expected_shape
    assert np.all(np.isfinite(values_np))
    assert np.linalg.norm(values_np[..., 1:]) > 0.0

for name, values in zip(("X", "Y", "Z"), (X, Y, Z)):
    values_np = np.asarray(values)

    assert values_np.shape == expected_shape
    assert np.all(np.isfinite(values_np))
    assert np.linalg.norm(values_np[..., 1:]) > 0.0

print("Walkthrough smoke test passed.")

Walkthrough smoke test passed.
